In [1]:
import sys
sys.path.append("..")

%load_ext autoreload 
%autoreload 2

In [2]:
from src.data import load_fd001, add_rul, FEATURES
import pandas as pd

In [3]:
train, test, rul = load_fd001()
train = add_rul(train, cap=125)

aus eda_01:

In [4]:
len(FEATURES)

14

In [5]:
X_train = train[FEATURES]
y_train = train["RUL"]
X_test = test.groupby("unit")[FEATURES].last()
y_test = rul["RUL"]

In [6]:
X_test.shape

(100, 14)

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [8]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

root_mean_squared_error(y_test, y_pred)

21.900213406890536

In [9]:
from xgboost import XGBRegressor
xgb_model = XGBRegressor(random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

In [10]:
root_mean_squared_error(y_test, y_pred_xgb)

18.177173614501953

In [11]:
y_pred_train_xgb = xgb_model.predict(X_train)
print(y_pred_train_xgb)
root_mean_squared_error(y_train, y_pred_train_xgb)

[119.55811   118.14523   121.27614   ...   5.387664    1.0365708
   2.2939532]


12.391884803771973

Anzeichen für Overfitting da RMSE der Testdaten bei 12,4 und bei den Testdaten bei 18,2 also ca 50% höher. Aufgrund unteschiedlicher Messdaten allerdings nicht vergleichbar 

In [12]:
y_pred_train = model.predict(X_train)
root_mean_squared_error(y_train, y_pred_train)

21.491018701515173

In [13]:
y_pred_train.min()

np.float64(-79.75576181896031)

In [14]:
y_pred_train.max()

np.float64(159.2387168622663)

In [15]:
print(f"Testtriebwerke mit RUL > 125: {(rul['RUL']>125).sum()}")
print(f"XGBoost Train: min {y_pred_train_xgb.min():.1f}, max {y_pred_train_xgb.max():.1f}")

Testtriebwerke mit RUL > 125: 11
XGBoost Train: min -2.2, max 134.5


In [18]:
print(f"XGBoost Train RMSE: {root_mean_squared_error(y_train, y_pred_train_xgb):.2f}")
print(f"LinReg Vorhersagen <0: {(y_pred_train < 0).sum()}")
print(f"LinReg Min: {y_pred_train.min()}")

XGBoost Train RMSE: 12.39
LinReg Vorhersagen <0: 465
LinReg Min: -79.75576181896031


## Ergebnisse der Baseline

| Modell | RMSE Train (alle Zyklen) | RMSE Test (letzter Zyklus, 100 Triebwerke) |
|---|---|---|
| Lineare Regression | 21,49 | 21,9 |
| XGBoost (Standardparameter) | 12,39 | 18,2 |

**Einordnung**
- **LinReg underfittet:** Train- und Test-RMSE liegen fast gleichauf. Das Modell hat sich nichts gemerkt, ist aber zu einfach. Eine lineare Funktion kann weder das Plateau bei 125 noch das Abknicken der Sensoren vor dem Ausfall abbilden.
- **Unbegrenzte Vorhersagen:** Auf Train sagt die LinReg Werte von −80 bis 159 voraus. 465 Zeilen erhalten eine physikalisch unmögliche negative Restlebensdauer, weil eine Ebene nach oben und unten nicht begrenzt ist.
- **XGBoost** senkt den Test-RMSE auf 18,2, das sind rund 17 % weniger. Seine Vorhersagen bleiben nahezu im Labelbereich (−2,2 bis 134,5). Die Blätter der Bäume enthalten Mittelwerte von Trainingslabels. Leichte Überschreitungen entstehen, weil Boosting die Korrekturen vieler Bäume addiert.
- **Kappung im Test:** 11 von 100 Testtriebwerken haben eine wahre RUL über 125. Ein auf das gekappte Label trainiertes Modell unterschätzt sie systematisch. Der Fehler geht in die sichere Richtung: Wartung zu früh statt zu spät.
- **Grenzen:** Der Test-RMSE beruht auf nur 100 Punkten und einem einzigen Split. Den belastbaren Vergleich und den Overfitting-Check für XGBoost liefert `03_validation` mit GroupKFold.